In [ ]:
import asyncio
import sys

from dotenv import dotenv_values
from langchain_community.vectorstores.neo4j_vector import Neo4jVector

from Database.transform import (
    awrite_concept_nodes,
    awrite_description_nodes,
    awrite_relationships,
    delete_all_in_batches,
    reformat_snomed,
)

In [ ]:
env = dotenv_values("YOUR_PATH_TO_ENV_FILE")  # Update this to your actual .env file path
neo4j_dict = {
        "uri": env.get('NEO4J_HOST'),
        "username": env.get('NEO4J_USER'),
        "password": env.get('NEO4J_PASSWORD'),
        "snomed_release": env.get('SNOMED_RELEASE_PATH'),
}
neo4j_uri = neo4j_dict["uri"]
neo4j_username = neo4j_dict["username"]
neo4j_secret = neo4j_dict["password"]

In [ ]:
from Database.embed_models import (
    AsyncPubMedBERTEmbedder, 
    AsyncSapBERTEmbedder, 
    AsyncSapUMLSEmbedder,
    AsyncBioClinicalBERTEmbedder, 
    AsyncGatorTronEmbedder,
    AsyncSapBERTpoolEmbedder)

In [ ]:
path = neo4j_dict["snomed_release"]
c_, s_, r_ = reformat_snomed(data_path=path)

In [ ]:
async def main():
      if sys.platform == "win32":
        asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())
      await awrite_concept_nodes(
          lf=c_, 
          neo4j_url=neo4j_uri, 
          neo4j_user=neo4j_username, 
          neo4j_password=neo4j_secret,
          batch_size = 10_000,
          )
      await awrite_description_nodes(
          lf= s_,
          embedding_model=embedding_model,
          neo4j_url=neo4j_uri,
          neo4j_user=neo4j_username,
          neo4j_password=neo4j_secret,
          batch_size = 10_000,
      )
      await awrite_relationships(r_, neo4j_uri, neo4j_username, neo4j_secret)


In [ ]:
def topk_accuracy_from_vector_search(
    gold_data,
    pred_data,
    vector_index,
    top_k_values=[1, 5, 10],
):
    """
    Evaluate top-k accuracy using vector similarity search.
    It uses the formal_term in predicted entities to query Neo4j vector index.

    Args:
        gold_data: list of dicts with 'entity' and 'Target code'
        pred_data: list of predicted entities (no 'tops', uses formal_term)
        vector_index: a Neo4jVector index instance (e.g., from_existing_index)
        top_k_values: list of k values to evaluate (default: [1, 5, 10])

    Returns:
        dict with top-k accuracy scores
    """
    assert len(gold_data) == len(pred_data), "Mismatch in number of records."

    topk_hits = {k: 0 for k in top_k_values}
    total_evaluated_spans = 0

    for gold_item, pred_entities in zip(gold_data, pred_data):
        gold_ents = {
            (e["start_index"], e["end_index"]): str(e["Target code"])
            for e in gold_item.get("entity", [])
        }

        
        # Keep track of spans processed in this record
        record_evaluated_spans = 0

        for pred in pred_entities:
            span = (pred["start_index"], pred["end_index"])
            if span not in gold_ents:
                continue

            gold_cid = gold_ents[span]
            query_text = pred.get("formal_term") or pred.get("term")

            if not query_text:
                continue

            # Vector search from Neo4j
            results = vector_index.similarity_search(query_text, k=max(top_k_values))
            top_candidates = [str(doc.metadata.get("concept_id")) for doc in results]
            
            # This is where we count the successes and failures for this span
            total_evaluated_spans += 1
            record_evaluated_spans += 1

            for k in top_k_values:
                if gold_cid in top_candidates[:k]:
                    topk_hits[k] += 1

    # FINAL SUMMARY LOG
#    print("\n" + "=" * 30)
#    print("      FINAL ACCURACY REPORT      ")
#    print("=" * 30)
    print(f"Total evaluated spans: {total_evaluated_spans}")

    results = {
        f"top{k}": topk_hits[k] / total_evaluated_spans if total_evaluated_spans else 0.0
        for k in top_k_values
    }

    # Format and print final results
    final_results_formatted = {
        f"TOP{k}": f"{(results[f'top{k}'] * 100):.2f}%"
        for k in top_k_values
    }
    
    #for k, v in final_results_formatted.items():
    #    print(f"  - {k} Accuracy: {v}")

    return final_results_formatted


In [ ]:
import json
gold_path = "DATA/test.json"
gemini_path = "DATA/predictions/gemini_pro.json"
claude_path = "DATA/predictions/claude.json"
gpt_path = "DATA/predictions/gpt41.json"

with open(gold_path, "r", encoding="utf-8") as f:
    gold_data = json.load(f)
with open(gemini_path, "r", encoding="utf-8") as f:
    gemini_data = json.load(f)
with open(gpt_path, "r", encoding="utf-8") as f:
    gpt_data = json.load(f)
with open(claude_path, "r", encoding="utf-8") as f:
    claude_data = json.load(f)

models = [
    ("Gemini", gemini_data),
    ("Claude", claude_data),
    ("GPT", gpt_data),
]

In [ ]:
embedding_model = AsyncPubMedBERTEmbedder()
neo4j_vector = Neo4jVector.from_existing_index(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_secret,
    embedding=embedding_model,
    database="neo4j",
    node_label="Concept",
    text_node_property="text",
    embedding_node_property="embedding",
    index_name="name_embedding"
)
print("\n" + "=" * 30)
print(f'Embedding model: {embedding_model.__class__.__name__}')
print("\n" + "=" * 30)
for name, pdata in models:
    res = topk_accuracy_from_vector_search(
        gold_data=gold_data,
        pred_data=pdata,
        vector_index=neo4j_vector,
        top_k_values=[1, 5, 10]
    )
    print(f'Evalution of {name}')
    print(res)
    print("\n" + "=" * 30)

In [ ]:
stats = delete_all_in_batches(
    uri=neo4j_uri,
    user=neo4j_username,
    password=neo4j_secret,
    batch_size=30000
)
print(stats)

In [ ]:
embedding_model = AsyncSapUMLSEmbedder()
await main()

In [ ]:
neo4j_vector = Neo4jVector.from_existing_index(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_secret,
    embedding=embedding_model,
    database="neo4j",
    node_label="Concept",
    text_node_property="text",
    embedding_node_property="embedding",
    index_name="name_embedding"
)
print("\n" + "=" * 30)
print(f'Embedding model: {embedding_model.__class__.__name__}')
print("\n" + "=" * 30)
for name, pdata in models:
    res = topk_accuracy_from_vector_search(
        gold_data=gold_data,
        pred_data=pdata,
        vector_index=neo4j_vector,
        top_k_values=[1, 5, 10]
    )
    print(f'Evalution of {name}')
    print(res)
    print("\n" + "=" * 30)

In [ ]:
stats = delete_all_in_batches(
    uri=neo4j_uri,
    user=neo4j_username,
    password=neo4j_secret,
    batch_size=30000
)
print(stats)
embedding_model = AsyncBioClinicalBERTEmbedder()
await main()


In [ ]:
neo4j_vector = Neo4jVector.from_existing_index(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_secret,
    embedding=embedding_model,
    database="neo4j",
    node_label="Concept",
    text_node_property="text",
    embedding_node_property="embedding",
    index_name="name_embedding"
)
print("\n" + "=" * 30)
print(f'Embedding model: {embedding_model.__class__.__name__}')
print("\n" + "=" * 30)
for name, pdata in models:
    res = topk_accuracy_from_vector_search(
        gold_data=gold_data,
        pred_data=pdata,
        vector_index=neo4j_vector,
        top_k_values=[1, 5, 10]
    )
    print(f'Evalution of {name}')
    print(res)
    print("\n" + "=" * 30)

In [ ]:
stats = delete_all_in_batches(
    uri=neo4j_uri,
    user=neo4j_username,
    password=neo4j_secret,
    batch_size=30000
)
print(stats)
embedding_model = AsyncGatorTronEmbedder()
await main()

In [ ]:
neo4j_vector = Neo4jVector.from_existing_index(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_secret,
    embedding=embedding_model,
    database="neo4j",
    node_label="Concept",
    text_node_property="text",
    embedding_node_property="embedding",
    index_name="name_embedding"
)
print("\n" + "=" * 30)
print(f'Embedding model: {embedding_model.__class__.__name__}')
print("\n" + "=" * 30)
for name, pdata in models:
    res = topk_accuracy_from_vector_search(
        gold_data=gold_data,
        pred_data=pdata,
        vector_index=neo4j_vector,
        top_k_values=[1, 5, 10]
    )
    print(f'Evalution of {name}')
    print(res)
    print("\n" + "=" * 30)

In [ ]:
stats = delete_all_in_batches(
    uri=neo4j_uri,
    user=neo4j_username,
    password=neo4j_secret,
    batch_size=30000
)
print(stats)
embedding_model = AsyncSapBERTpoolEmbedder()
await main()

In [ ]:
neo4j_vector = Neo4jVector.from_existing_index(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_secret,
    embedding=embedding_model,
    database="neo4j",
    node_label="Concept",
    text_node_property="text",
    embedding_node_property="embedding",
    index_name="name_embedding"
)
print("\n" + "=" * 30)
print(f'Embedding model: {embedding_model.__class__.__name__}')
print("\n" + "=" * 30)
for name, pdata in models:
    res = topk_accuracy_from_vector_search(
        gold_data=gold_data,
        pred_data=pdata,
        vector_index=neo4j_vector,
        top_k_values=[1, 5, 10]
    )
    print(f'Evalution of {name}')
    print(res)
    print("\n" + "=" * 30)

In [ ]:
stats = delete_all_in_batches(
    uri=neo4j_uri,
    user=neo4j_username,
    password=neo4j_secret,
    batch_size=30000
)
print(stats)
embedding_model = AsyncSapBERTEmbedder()
await main()

In [ ]:
neo4j_vector = Neo4jVector.from_existing_index(
    url=neo4j_uri,
    username=neo4j_username,
    password=neo4j_secret,
    embedding=embedding_model,
    database="neo4j",
    node_label="Concept",
    text_node_property="text",
    embedding_node_property="embedding",
    index_name="name_embedding"
)
print("\n" + "=" * 30)
print(f'Embedding model: {embedding_model.__class__.__name__}')
print("\n" + "=" * 30)
for name, pdata in models:
    res = topk_accuracy_from_vector_search(
        gold_data=gold_data,
        pred_data=pdata,
        vector_index=neo4j_vector,
        top_k_values=[1, 5, 10]
    )
    print(f'Evalution of {name}')
    print(res)
    print("\n" + "=" * 30)